<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task3/Task3_Model4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#Task 3 (Part 2) — Logistic Regression & Decision Tree

#import and start pyspark
!pip install pyspark -q

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [7]:
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# Load preprocessed train/test data saved in Task 2
train_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet'
)
test_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet'
)

print(f"Training rows: {train_df.count():,}")
print(f"Testing rows: {test_df.count():,}")

# Use a smaller sample for model tuning - due to RAM issue
train_sample = train_df.sample(0.03, seed=42)
test_sample = test_df.sample(0.03, seed=42)

print(f"Training sample rows: {train_sample.count():,}")
print(f"Testing sample rows: {test_sample.count():,}")

print("Data loaded successfully!")

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training rows: 1,469,696
Testing rows: 367,299
Training sample rows: 43,683
Testing sample rows: 11,199
Data loaded successfully!


In [8]:
# ── MODEL 4: DECISION TREE CLASSIFIER
# Predicts whether a prescription belongs to the HIGH_COST category.

# Decision Trees create a series of if-then rules to classify data,
# making them one of the most interpretable machine learning models.

from pyspark.ml.classification import DecisionTreeClassifier

# Defining
# impurity='gini' measures how well a split separates the classes.
# Lower impurity means the resulting groups contain more similar labels.
dt_Model = DecisionTreeClassifier(
    featuresCol='scaled_features',
    labelCol='HIGH_COST',
    seed=42, #ensures the results can be reproduced.
    impurity='gini'
)

print("Decision Tree Classifier defined!")
print("Target variable: HIGH_COST")
print("Impurity measure: Gini")

Decision Tree Classifier defined!
Target variable: HIGH_COST
Impurity measure: Gini


In [12]:
# HYPERPARAMETER TUNING
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Create Decision Tree's own AUC-ROC evaluator
# (instead of reusing Logistic Regression's, which doesn't exist in this file)
dt_evaluator = BinaryClassificationEvaluator(
    labelCol='HIGH_COST',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
# maxDepth controls how many levels the tree can grow.
# Larger values allow more complex decision rules but may
# increase the risk of overfitting.

# minInstancesPerNode specifies the minimum number of records
# required in a leaf node. Smaller values allow finer splits,
# while larger values create simpler trees.

dt_grid = (
    ParamGridBuilder()
    .addGrid(dt_Model.maxDepth, [5, 8])
    .addGrid(dt_Model.minInstancesPerNode, [1, 5])
    .build()
)

# Reuse the AUC-ROC evaluator from Logistic Regression.
# This allows consistent comparison between classification models.

# AUC-ROC measures how effectively the model separates
# HIGH_COST prescriptions from non-HIGH_COST prescriptions.
# Values closer to 1 indicate stronger classification ability.

dt_cv = CrossValidator(
    estimator=dt_Model,
    estimatorParamMaps=dt_grid,
    evaluator=dt_evaluator,
    numFolds=2,      # reduced folds to minimise memory usage
    seed=42,         # ensures reproducible results
    parallelism=1    # train one model at a time to avoid RAM issues
)

print("Decision Tree CrossValidator configured!")
print(f"Parameter combinations: {len(dt_grid)}")
print("Parameters being tuned:")
print("  maxDepth: [5, 8]")
print("  minInstancesPerNode: [1, 5]")
print("Evaluation metric: AUC-ROC")
print("Cross-validation folds: 2")

Decision Tree CrossValidator configured!
Parameter combinations: 4
Parameters being tuned:
  maxDepth: [5, 8]
  minInstancesPerNode: [1, 5]
Evaluation metric: AUC-ROC
Cross-validation folds: 2
